In [1]:
import requests
import csv
from datetime import datetime

API_KEY = "YOUR_OPENWEATHER_API_KEY"   # get it from openweathermap.org
CITY = "Cairo"
URL = f"https://api.openweathermap.org/data/2.5/weather?q={CITY}&appid={API_KEY}&units=metric"

def get_weather():
    resp = requests.get(URL, timeout=10)
    resp.raise_for_status()
    data = resp.json()
    return {
        "city": data["name"],
        "temperature": data["main"]["temp"],
        "humidity": data["main"]["humidity"],
        "description": data["weather"][0]["description"],
        "timestamp": datetime.now().isoformat(timespec="seconds")
    }

def save_to_csv(weather, filename="weather.csv"):
    fieldnames = ["city", "temperature", "humidity", "description", "timestamp"]
    try:
        # Append if file exists, else create with header
        with open(filename, "a", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            if f.tell() == 0:  # if file is empty, write header
                writer.writeheader()
            writer.writerow(weather)
        print("✅ Weather saved to", filename)
    except Exception as e:
        print("❌ Error saving CSV:", e)

if __name__ == "__main__":
    weather = get_weather()
    print("Fetched:", weather)
    save_to_csv(weather)


HTTPError: 401 Client Error: Unauthorized for url: https://api.openweathermap.org/data/2.5/weather?q=Cairo&appid=YOUR_OPENWEATHER_API_KEY&units=metric

In [2]:
pip install requests

Note: you may need to restart the kernel to use updated packages.


In [3]:
pip install beautifulsoup

  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'error'
Note: you may need to restart the kernel to use updated packages.


  error: subprocess-exited-with-error
  
  python setup.py egg_info did not run successfully.
  exit code: 1
  
  [16 lines of output]
  Traceback (most recent call last):
    File "<string>", line 2, in <module>
      exec(compile('''
      ~~~~^^^^^^^^^^^^
      # This is <pip-setuptools-caller> -- a caller that pip uses to run setup.py
      ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
      ...<32 lines>...
      exec(compile(setup_py_code, filename, "exec"))
      ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
      ''' % ('C:\\Users\\omark\\AppData\\Local\\Temp\\pip-install-8vm74snr\\beautifulsoup_23380929755744cbae49c7786934ebf5\\setup.py',), "<pip-setuptools-caller>", "exec"))
      ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    File "<pip-setuptools-caller>", line 35, in <module>
    File "C:\Users\omark\AppData\Local\Temp\p

In [1]:
pip install beautifulsoup


  Using cached BeautifulSoup-3.2.2.tar.gz (32 kB)
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'error'


  error: subprocess-exited-with-error
  
  python setup.py egg_info did not run successfully.
  exit code: 1
  
  [16 lines of output]
  Traceback (most recent call last):
    File "<string>", line 2, in <module>
      exec(compile('''
      ~~~~^^^^^^^^^^^^
      # This is <pip-setuptools-caller> -- a caller that pip uses to run setup.py
      ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
      ...<32 lines>...
      exec(compile(setup_py_code, filename, "exec"))
      ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
      ''' % ('C:\\Users\\omark\\AppData\\Local\\Temp\\pip-install-xrvj24in\\beautifulsoup_3851e3deeeab48579992757f54519be2\\setup.py',), "<pip-setuptools-caller>", "exec"))
      ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    File "<pip-setuptools-caller>", line 35, in <module>
    File "C:\Users\omark\AppData\Local\Temp\p

In [2]:
import beautifulsoup

ModuleNotFoundError: No module named 'beautifulsoup'

In [3]:
from bs4 import BeautifulSoup

In [4]:
import requests
from bs4 import BeautifulSoup
import csv
from datetime import datetime

def fetch_weather(city="Cairo"):
    url = f"https://wttr.in/{city}?format=v2"  # formatted weather
    headers = {"User-Agent": "Mozilla/5.0 (compatible; MyWeatherScraper/1.0)"}
    resp = requests.get(url, headers=headers, timeout=10)
    resp.raise_for_status()
    
    # wttr.in returns plain text if you use ?format=, but let’s pretend we parse HTML
    soup = BeautifulSoup(resp.text, "html.parser")
    
    # Example: here we just get raw text (you’d normally inspect the HTML to find exact classes)
    text_weather = soup.get_text().strip()
    
    # For demonstration, we’ll keep only first line
    first_line = text_weather.splitlines()[0] if text_weather else "N/A"
    
    return {
        "city": city,
        "weather": first_line,
        "timestamp": datetime.now().isoformat(timespec="seconds")
    }

def save_csv(weather, filename="weather_bs.csv"):
    fieldnames = ["city", "weather", "timestamp"]
    with open(filename, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        if f.tell() == 0:  # write header if file is empty
            writer.writeheader()
        writer.writerow(weather)

if __name__ == "__main__":
    data = fetch_weather("Cairo")
    print("Fetched:", data)
    save_csv(data)
    print("✅ Saved to weather_bs.csv")


Fetched: {'city': 'Cairo', 'weather': 'Weather report for Cairo', 'timestamp': '2025-10-01T00:42:28'}
✅ Saved to weather_bs.csv


In [5]:
"""
Fetch 14-day daily forecasts from Open-Meteo for many cities and save to CSV.
Requirements: pip install requests
Run: python fetch_weather_openmeteo.py
"""

import requests
import csv
import time
from datetime import datetime

# --- CONFIG ---
OUTPUT_CSV = "weather_14days.csv"
FORECAST_DAYS = 14
DELAY_BETWEEN_CALLS = 1.0  # seconds (politeness)
# Daily variables to request
DAILY_VARS = [
    "temperature_2m_max",
    "temperature_2m_min",
    "precipitation_sum",
    "weathercode",
    "sunrise",
    "sunset"
]

# List of cities (city, latitude, longitude)
# ~38 cities -> 38 * 14 = 532 records (>= 500)
CITIES = [
    ("Cairo, Egypt", 30.0444, 31.2357),
    ("London, UK", 51.5074, -0.1278),
    ("Paris, France", 48.8566, 2.3522),
    ("Berlin, Germany", 52.52, 13.4050),
    ("Madrid, Spain", 40.4168, -3.7038),
    ("Rome, Italy", 41.9028, 12.4964),
    ("Athens, Greece", 37.9838, 23.7275),
    ("Ankara, Turkey", 39.9334, 32.8597),
    ("Moscow, Russia", 55.7558, 37.6173),
    ("Washington, USA", 38.9072, -77.0369),
    ("Ottawa, Canada", 45.4215, -75.6972),
    ("Canberra, Australia", -35.2809, 149.1300),
    ("Wellington, New Zealand", -41.2865, 174.7762),
    ("Tokyo, Japan", 35.6762, 139.6503),
    ("Beijing, China", 39.9042, 116.4074),
    ("New Delhi, India", 28.6139, 77.2090),
    ("Islamabad, Pakistan", 33.6844, 73.0479),
    ("Riyadh, Saudi Arabia", 24.7136, 46.6753),
    ("Dubai, UAE", 25.2048, 55.2708),
    ("Johannesburg, South Africa", -26.2041, 28.0473),
    ("Nairobi, Kenya", -1.2921, 36.8219),
    ("Lagos, Nigeria", 6.5244, 3.3792),
    ("Buenos Aires, Argentina", -34.6037, -58.3816),
    ("Sao Paulo, Brazil", -23.5505, -46.6333),
    ("Mexico City, Mexico", 19.4326, -99.1332),
    ("Santiago, Chile", -33.4489, -70.6693),
    ("Jakarta, Indonesia", -6.2088, 106.8456),
    ("Seoul, South Korea", 37.5665, 126.9780),
    ("Singapore", 1.3521, 103.8198),
    ("Bangkok, Thailand", 13.7563, 100.5018),
    ("Hanoi, Vietnam", 21.0278, 105.8342),
    ("Tehran, Iran", 35.6892, 51.3890),
    ("Baghdad, Iraq", 33.3152, 44.3661),
    ("Kyiv, Ukraine", 50.4501, 30.5234),
    ("Budapest, Hungary", 47.4979, 19.0402),
    ("Stockholm, Sweden", 59.3293, 18.0686),
    ("Lisbon, Portugal", 38.7223, -9.1393),
]

# --- Helpers ---
def build_url(lat, lon, timezone="auto"):
    base = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude": lat,
        "longitude": lon,
        "daily": ",".join(DAILY_VARS),
        "forecast_days": FORECAST_DAYS,
        "timezone": timezone
    }
    # build query string
    qs = "&".join(f"{k}={requests.utils.quote(str(v))}" for k, v in params.items())
    return f"{base}?{qs}"

def fetch_forecast(lat, lon):
    url = build_url(lat, lon)
    resp = requests.get(url, timeout=15)
    resp.raise_for_status()
    return resp.json()

# --- Main ---
def main():
    # prepare CSV
    fieldnames = [
        "city", "latitude", "longitude", "date",
        "temp_max_C", "temp_min_C", "precipitation_mm",
        "weathercode", "sunrise", "sunset", "timezone",
        "fetched_at"
    ]
    rows_written = 0

    with open(OUTPUT_CSV, "w", newline="", encoding="utf-8") as csvfile:
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        writer.writeheader()

        for city_name, lat, lon in CITIES:
            try:
                print(f"[{datetime.now().isoformat()}] Fetching {city_name} ({lat},{lon})...")
                data = fetch_forecast(lat, lon)
                tz = data.get("timezone", "")
                # daily arrays
                daily = data.get("daily", {})
                dates = daily.get("time", [])
                tmax = daily.get("temperature_2m_max", [])
                tmin = daily.get("temperature_2m_min", [])
                precip = daily.get("precipitation_sum", [])
                wcode = daily.get("weathercode", [])
                sunrise = daily.get("sunrise", [])
                sunset = daily.get("sunset", [])

                for i, d in enumerate(dates):
                    row = {
                        "city": city_name,
                        "latitude": lat,
                        "longitude": lon,
                        "date": d,
                        "temp_max_C": tmax[i] if i < len(tmax) else "",
                        "temp_min_C": tmin[i] if i < len(tmin) else "",
                        "precipitation_mm": precip[i] if i < len(precip) else "",
                        "weathercode": wcode[i] if i < len(wcode) else "",
                        "sunrise": sunrise[i] if i < len(sunrise) else "",
                        "sunset": sunset[i] if i < len(sunset) else "",
                        "timezone": tz,
                        "fetched_at": datetime.now().isoformat(timespec="seconds")
                    }
                    writer.writerow(row)
                    rows_written += 1

                print(f"  -> wrote {len(dates)} rows for {city_name}. Total so far: {rows_written}")
            except Exception as e:
                print(f"Error fetching {city_name}: {e}")

            time.sleep(DELAY_BETWEEN_CALLS)

    print(f"Done. Wrote {rows_written} rows to {OUTPUT_CSV}")

if __name__ == "__main__":
    main()


[2025-10-01T00:48:44.201050] Fetching Cairo, Egypt (30.0444,31.2357)...
  -> wrote 14 rows for Cairo, Egypt. Total so far: 14
[2025-10-01T00:48:45.759480] Fetching London, UK (51.5074,-0.1278)...
  -> wrote 14 rows for London, UK. Total so far: 28
[2025-10-01T00:48:47.282374] Fetching Paris, France (48.8566,2.3522)...
  -> wrote 14 rows for Paris, France. Total so far: 42
[2025-10-01T00:48:48.715140] Fetching Berlin, Germany (52.52,13.405)...
  -> wrote 14 rows for Berlin, Germany. Total so far: 56
[2025-10-01T00:48:50.150626] Fetching Madrid, Spain (40.4168,-3.7038)...
  -> wrote 14 rows for Madrid, Spain. Total so far: 70
[2025-10-01T00:48:51.564024] Fetching Rome, Italy (41.9028,12.4964)...
  -> wrote 14 rows for Rome, Italy. Total so far: 84
[2025-10-01T00:48:53.015444] Fetching Athens, Greece (37.9838,23.7275)...
  -> wrote 14 rows for Athens, Greece. Total so far: 98
[2025-10-01T00:48:54.453534] Fetching Ankara, Turkey (39.9334,32.8597)...
  -> wrote 14 rows for Ankara, Turkey. T